# 🚀 AI Meets Control Theory (AIMCT) — Interactive Guided Tour

**A rigorous, code-first bridge from classical feedback control to modern machine learning, physics-informed dynamics, and safe reinforcement learning.**

This tour walks through the five core pillars of the library:
1. **Dynamical Systems Modeling**: Physical models (`MassSpringDamper`, `CartPole`, `PlanarQuadrotor`)
2. **State-Space & Optimal Control**: PID, LQR, and from-scratch Riccati solvers (`solve_care`)
3. **Nonlinear & Underactuated Hybrid Control**: Spong energy-shaping swing-up with LQR handoff
4. **Constrained Optimal Control**: Receding-horizon Model Predictive Control (`LinearMPC`)
5. **Automated Benchmark Harness**: Multi-controller comparison and scoring (`aimct.benchmarks.compare`)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply the unified AIMCT publication plotting style
from aimct.plot_style import set_aimct_style
set_aimct_style()

print("AIMCT environment loaded successfully!")

## 1. ⚙️ Dynamical Systems from First Principles

Every system in `aimct.systems` implements the `DynamicalSystem` interface, exposing exact nonlinear differential equations $\dot{x} = f(t, x, u)$ and analytical state-space linearizations $(A, B)$.

In [ ]:
from aimct.systems import MassSpringDamper, CartPole, PlanarQuadrotor

# 1. Mass-Spring-Damper (1-DOF linear mechanical oscillator)
msd = MassSpringDamper(m=1.0, c=0.4, k=1.0)
A_msd, B_msd = msd.linearize()
print("Mass-Spring-Damper A matrix:\n", A_msd)
print("Mass-Spring-Damper B matrix:\n", B_msd)

# 2. Cart-Pole (Nonlinear underactuated inverted pendulum)
cartpole = CartPole(m_cart=1.0, m_pole=0.1, length=0.5)
print(f"\nCart-Pole: {cartpole.n_states} states, {cartpole.n_inputs} input.")
A_cp, B_cp = cartpole.linearize(x_eq=[0, 0, 0, 0])  # Linearization about upright equilibrium
open_loop_poles = np.linalg.eigvals(A_cp)
print("Cart-Pole open-loop poles about upright:\n", open_loop_poles)

## 2. 🎛️ Linear Quadratic Regulator (LQR) & Classical Control

We can design optimal state feedback gains $K = R^{-1} B^T P$ using our from-scratch Continuous Algebraic Riccati Equation (CARE) solver:
$$A^T P + P A - P B R^{-1} B^T P + Q = 0$$

In [ ]:
from aimct.controllers import LQR, StateFeedback, PID
from aimct.simulate import simulate

# Design LQR for Mass-Spring-Damper
Q = np.diag([10.0, 1.0])   # Penalize position error and velocity
R = np.array([[0.1]])      # Penalize control effort

lqr = LQR(A_msd, B_msd, Q, R)
print(f"Optimal LQR Feedback Gain K: {lqr.K.ravel()}")
print(f"Closed-loop eigenvalues: {np.linalg.eigvals(A_msd - B_msd @ lqr.K)}")

# Wrap inside StateFeedback controller with a step reference x_ref = [1.0, 0.0]
ctrl_lqr = StateFeedback(lqr.K, x_ref=np.array([1.0, 0.0]))

# Simulate closed-loop response
traj_lqr = simulate(msd, ctrl_lqr, x0=[0.0, 0.0], dt=0.002, t_final=8.0, u_bounds=(-20.0, 20.0))
print(f"Simulated {len(traj_lqr.t)} steps. Final position: {traj_lqr.x[-1, 0]:.4f}")

## 3. 🔄 Nonlinear Underactuated Hybrid Control: Cart-Pole Swing-Up

When initialized hanging downward ($x_0 = [0, 0, \pi, 0]$), pure linear LQR cannot stabilize the pole. We compose:
1. **Spong Partial Feedback Linearization (Energy Shaping)** to pump mechanical energy toward the upright separatrix.
2. **Hysteresis State Machine** that catches the pole inside a $|\theta| \le 0.35\text{ rad}$ window and hands off to LQR.

In [ ]:
from aimct.controllers.swingup import EnergyShapingSwingUp, HybridSwingUpLQR

# Upright LQR balance law
lqr_cp = LQR(A_cp, B_cp, Q=np.diag([10.0, 1.0, 100.0, 10.0]), R=np.array([[0.1]]))

# Energy shaping pump law
swingup_law = EnergyShapingSwingUp(cartpole, k_energy=15.0, k_cart=2.0, k_cart_rate=1.5, u_max=20.0)

# Hybrid hysteresis controller
hybrid_ctrl = HybridSwingUpLQR(swingup_law, lqr_cp, capture_angle=0.35, capture_rate=1.5, release_angle=0.60)

# Simulate full swing-up from downward hanging rest x0 = [0, 0, pi, 0]
traj_cp = simulate(cartpole, hybrid_ctrl, x0=[0.0, 0.0, np.pi, 0.0], dt=0.002, t_final=10.0, u_bounds=(-20.0, 20.0))

final_th_err = np.abs(np.arctan2(np.sin(traj_cp.x[-1, 2]), np.cos(traj_cp.x[-1, 2])))
print("Cart-Pole Swing-Up completed!")
print(f"- Final angle error: {final_th_err:.2e} rad")
print(f"- Max cart displacement: {np.max(np.abs(traj_cp.x[:, 0])):.2f} m (well within rail limits)")
print(f"- Handoff from swing-up to LQR balance occurred at step: {hybrid_ctrl.switch_steps}")

## 4. 📊 High-Fidelity Visualizations

Let's plot the full 4-panel trajectory of the Cart-Pole swing-up using the standard AIMCT visualization layout:

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))

t = traj_cp.t
x = traj_cp.x
u = traj_cp.u

# (a) Cart position and rail limits
ax[0, 0].plot(t, x[:, 0], color="#1f77b4", lw=2, label="Cart position $x$")
ax[0, 0].axhline(2.4, color="#d62728", ls="--", lw=1.5, label="Rail limit $\pm 2.4$ m")
ax[0, 0].axhline(-2.4, color="#d62728", ls="--", lw=1.5)
ax[0, 0].set_ylabel("Position [m]")
ax[0, 0].set_title("(a) Cart Translation", fontweight="bold")
ax[0, 0].legend(fontsize=8, loc="upper right")
ax[0, 0].grid(True, alpha=0.3)

# (b) Pendulum angle
th_wrapped = np.arctan2(np.sin(x[:, 2]), np.cos(x[:, 2]))
ax[0, 1].plot(t, th_wrapped * 180 / np.pi, color="#2ca02c", lw=2, label="Pole angle $\theta$")
ax[0, 1].axhline(0, color="#333", ls=":", lw=1)
ax[0, 1].set_ylabel("Angle [deg]")
ax[0, 1].set_title("(b) Pole Angle from Upright", fontweight="bold")
ax[0, 1].legend(fontsize=8, loc="upper right")
ax[0, 1].grid(True, alpha=0.3)

# (c) Control force input and actuator saturation
ax[1, 0].plot(t, u[:, 0], color="#ff7f0e", lw=1.8, label="Force $F(t)$")
ax[1, 0].axhline(20.0, color="#d62728", ls="--", lw=1, label="Saturation $\pm 20$ N")
ax[1, 0].axhline(-20.0, color="#d62728", ls="--", lw=1)
ax[1, 0].set_xlabel("Time [s]")
ax[1, 0].set_ylabel("Control Force [N]")
ax[1, 0].set_title("(c) Control Effort", fontweight="bold")
ax[1, 0].legend(fontsize=8, loc="upper right")
ax[1, 0].grid(True, alpha=0.3)

# (d) Phase portrait: theta vs theta_dot
ax[1, 1].plot(th_wrapped, x[:, 3], color="#9467bd", lw=1.5)
ax[1, 1].plot([0], [0], "r*", ms=12, label="Upright attractor $(0, 0)$")
ax[1, 1].set_xlabel("Pole Angle $\theta$ [rad]")
ax[1, 1].set_ylabel("Angular Velocity $\dot{\theta}$ [rad/s]")
ax[1, 1].set_title("(d) Phase Portrait", fontweight="bold")
ax[1, 1].legend(fontsize=8, loc="upper right")
ax[1, 1].grid(True, alpha=0.3)

fig.suptitle("AIMCT Tour: Cart-Pole Energy Swing-Up & LQR Balance", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## 5. 🏆 Automated Benchmark Comparison Harness

`aimct.benchmarks.compare` provides one-line comparative evaluations across multiple controller families on any plant.

In [ ]:
from aimct.benchmarks import compare
from aimct.controllers import PID, LinearMPC

# Position feedback wrapper for PID on 1-DOF position channel
class PositionPID(PID):
    def update(self, measurement, dt):
        pos = measurement[0] if hasattr(measurement, '__len__') else measurement
        return np.array([super().update(pos, dt)])

# Set up 3 controllers for Mass-Spring-Damper setpoint tracking
pid_ctrl = PositionPID(kp=15.0, ki=5.0, kd=3.0, output_limits=(-20.0, 20.0), setpoint=1.0)
lqr_ctrl = StateFeedback(lqr.K, x_ref=np.array([1.0, 0.0]))
mpc_ctrl = LinearMPC(A_msd, B_msd, Q=Q, R=R, N=15, u_bounds=(-20.0, 20.0), x_ref=np.array([1.0, 0.0]))

# Run automated comparison
result = compare(
    system=msd,
    controllers={
        "PID": pid_ctrl,
        "LQR": lqr_ctrl,
        "Linear MPC": mpc_ctrl,
    },
    x0=[0.0, 0.0],
    reference=1.0,
    t_final=6.0,
    dt=0.002,
    u_bounds=(-20.0, 20.0),
)

# Display comparison markdown table
print(result.to_markdown())

---
### 🎓 Next Steps & Deeper Modules
- **Module 01–05**: Classical Control, Frequency Margins, State Estimation (Kalman/EKF/UKF), and LQR
- **Module 06**: Data-Driven Dynamics Discovery (SINDy, Neural ODEs)
- **Module 07**: Reinforcement Learning for Continuous Control (PPO, DDPG vs LQR Baselines)
- **Module 08**: Safe AI Control (Control Barrier Functions & Differentiable MPC)
- **Module 09**: Intelligent Control Challenge & Quadrotor Five-Way Bake-Off